In [1]:
%pwd

'd:\\Siam\\Chicken-Disease-Classification\\research'

In [2]:
import os

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Siam\\Chicken-Disease-Classification'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class PrepareCallbacksConfig:
    root_dir: Path
    tensorboard_root_log_dir: Path
    checkpoint_model_filepath: Path

In [6]:
from chicken_disease_classification.constants import *
from chicken_disease_classification.utils.common import read_yaml, create_directories

In [12]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config["artifacts_root"]])

    def get_prepare_callbacks_config(self) -> PrepareCallbacksConfig:
        config = self.config["prepare_callbacks"]
        model_ckpt_dir = os.path.dirname(config["checkpoint_model_filepath"])
        create_directories([
            Path(model_ckpt_dir),
            Path(config["tensorboard_root_log_dir"])
        ])
        prepare_callbacks_config = PrepareCallbacksConfig(
            root_dir=Path(config["root_dir"]),
            tensorboard_root_log_dir=Path(config["tensorboard_root_log_dir"]),
            checkpoint_model_filepath=Path(config["checkpoint_model_filepath"])
        )
        return prepare_callbacks_config

In [8]:
import os
import sys
import tensorflow as tf
from chicken_disease_classification.logger import logging
from chicken_disease_classification.exception import CustomException
import time

In [9]:
class PrepareCallbacksTrainingPipeline:
    def __init__(self, config: PrepareCallbacksConfig):
        self.config = config

    
    @property
    def _create_tb_callback(self):
        timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
        tb_run_log_dir = os.path.join(self.config.tensorboard_root_log_dir, timestamp)
        tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=tb_run_log_dir)
        return tensorboard_callback
    
    @property
    def _create_checkpoint_callback(self):
        checkpoint_dir = os.path.dirname(self.config.checkpoint_model_filepath)
        create_directories([checkpoint_dir])
        checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
            filepath=self.config.checkpoint_model_filepath,
            save_best_only=True,
        )
        return checkpoint_callback
    
    def get_tb_ckpt_callbacks(self):
        tb_callback = self._create_tb_callback
        ckpt_callback = self._create_checkpoint_callback
        return [
            tb_callback, 
            ckpt_callback
        ]

In [13]:
try:
    config = ConfigurationManager()
    prepare_callbacks_config = config.get_prepare_callbacks_config()
    prepare_callbacks = PrepareCallbacksTrainingPipeline(config=prepare_callbacks_config)
    callback_list = prepare_callbacks.get_tb_ckpt_callbacks()
except Exception as e:
    raise CustomException(e, sys)

[2026-05-10 15:48:09,574: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-10 15:48:09,576: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-10 15:48:09,577: INFO: common: created directory at: artifacts]
[2026-05-10 15:48:09,579: INFO: common: created directory at: artifacts\prepare_callbacks\checkpoint_dir]
[2026-05-10 15:48:09,580: INFO: common: created directory at: artifacts\prepare_callbacks\tensorboard_log_dir]
[2026-05-10 15:48:09,582: INFO: common: created directory at: artifacts\prepare_callbacks\checkpoint_dir]
